# Compute final quantitative metrics

This notebook aggregates the raw `final_results.pkl` files written by
`evaluate_SD_DSCM.py` into the tables reported for each dataset. It **merges**
the previous per-dataset notebooks (`vis_celeba`, `vis_celebahq`,
`vis_celeba_ablation`, `vis_celebahq_fulltest`, `vis_adni`, `vis_pendulum`).

All metric logic lives in `metrics_utils.py` — the cells below just point each
dataset section at its run directory and print the table. Edit `root_dir` in a
section to match your own `--output-root` / run name.

Run directory layout:
```
<output_root>/<dataset>/<run_name>/<do_attr>/final_results.pkl
```


In [ ]:
import os, sys, pickle
import numpy as np

# Make `ctf_datasets` importable (needed only by the ADNI section) and pull in
# the shared metric helpers. _paths bootstraps the project packages.
sys.path.append("..")          # methods/deepscm  -> exposes _paths
sys.path.append("../../../")   # counterfactual_benchmark package root (ctf_datasets, models, ...)
try:
    from _paths import bootstrap
    bootstrap()
except Exception as e:
    print("path bootstrap skipped:", e)

from metrics_utils import (
    evaluate_by_batch,            # celeba / celeba-HQ (binary F1 + accuracy)
    evaluate_adni_by_batch,       # ADNI (mixed classification + regression)
    evaluate_regression_by_batch, # pendulum (L1 regression)
    effectiveness_table,
    lpips_summary,
    list_do_folders,
    load_results,
)


## CelebA (complex graph)

Effectiveness (F1 / accuracy per `do(attr)`) and LPIPS identity distance.
Replaces `vis_celeba.ipynb`. For the ablation runs, just point `root_dir` at the
`saved_ablation/...` directory (this replaces `vis_celeba_ablation.ipynb`).

In [ ]:
root_dir = "../saved_benchmark/celeA_complex/DSCM_effectiveness_step100.0"
attr_keys = ['Young', 'Male', 'No_Beard', 'Bald']
do_folders = list_do_folders(root_dir, candidates=attr_keys)

# Effectiveness table
for target_attr in attr_keys:
    for do_attr in do_folders:
        loaded = load_results(os.path.join(root_dir, do_attr, "final_results.pkl"))
        if loaded is None:
            continue
        scores = evaluate_by_batch(loaded).get(target_attr)
        if scores:
            print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']}, Acc = {scores['Accuracy']}")

# LPIPS identity distance per do(attr)
print()
for do_attr, dists in lpips_summary(root_dir, do_folders).items():
    if 'lpips' in dists:
        print(f"do({do_attr}) lpips distance = {dists['lpips']:.4f}")


## CelebA-HQ (simple graph)

Effectiveness on `Smiling` / `Eyeglasses`. Replaces `vis_celebahq.ipynb`.
`evaluate_by_batch` applies the calibrated thresholds (`Smiling`=0.136,
`Eyeglasses`=0.159) from `metrics_utils.DEFAULT_THRESHOLDS`.

In [ ]:
root_dir = "../saved_benchmark/celebahq_simple/DSCM_effectiveness_step100.0"
target_attrs = ["Smiling", "Eyeglasses"]

table = effectiveness_table(root_dir, target_attrs=target_attrs)
for target_attr, by_do in table.items():
    print(f"=== target: {target_attr} ===")
    for do_attr, scores in by_do.items():
        print(f"  do({do_attr}): F1 = {scores['F1-score']}, Acc = {scores['Accuracy']}")

# LPIPS identity distance per do(attr)
print()
for do_attr, dists in lpips_summary(root_dir).items():
    if 'lpips' in dists:
        print(f"do({do_attr}) lpips distance = {dists['lpips']:.4f}")


### CelebA-HQ reverse / full-test metrics

For `--reverse` runs, each `final_results.pkl` also stores IDP / reverse /
composition L1+LPIPS arrays. Replaces `vis_celebahq_fulltest.ipynb`.

In [ ]:
# point this at a run produced with `--reverse`
reverse_root = "../saved_benchmark_full/celebahq_simple/DSCM_effectiveness_step100.0_reverse"
for do_attr, dists in lpips_summary(reverse_root).items():
    print(f"do({do_attr}):")
    for k in ['IDP_l1', 'IDP_lpips', 'reverse_l1', 'reverse_lpips', 'compos_l1', 'compos_lpips']:
        if k in dists:
            print(f"    {k} = {dists[k]:.4f}")


## ADNI

Mixed classification (`apoE`, `sex`, `slice`) + regression (`age`, `brain_vol`,
`vent_vol`). Replaces `vis_adni.ipynb`. Needs `torch` / `torchmetrics` and the
ADNI label codecs (handled inside `evaluate_adni_by_batch`).

In [ ]:
root_dir = "../saved_benchmark/ADNI/DSCM_effectiveness_step100.0_scale1.0"
attr_keys = ['apoE', 'age', 'sex', 'brain_vol', 'vent_vol', 'slice']
batch_size = 256
do_folders = list_do_folders(root_dir, candidates=attr_keys)

for target_attr in attr_keys:
    for do_attr in do_folders:
        loaded = load_results(os.path.join(root_dir, do_attr, "final_results.pkl"))
        if loaded is None:
            continue
        scores = evaluate_adni_by_batch(loaded, batch_size=batch_size).get(target_attr)
        if not scores:
            continue
        if 'F1-score' in scores:
            print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']} ± {scores.get('F1-std', 0)}, "
                  f"Acc = {scores['Accuracy']} ± {scores.get('Acc-std', 0)}")
        else:
            print(f"do({do_attr})\t{target_attr}: L1-loss = {scores['L1-loss']} ± {scores.get('L1-std', 0)}")


## Pendulum

All attributes are continuous → L1 error. Replaces `vis_pendulum.ipynb`.

In [ ]:
root_dir = "../saved_benchmark/pendulum/DSCM_effectiveness_step50.0_scale1.0"
attr_keys = ["pendulum", "light", "shadow_length", "shadow_position"]
do_folders = list_do_folders(root_dir, candidates=attr_keys)

for target_attr in attr_keys:
    for do_attr in do_folders:
        loaded = load_results(os.path.join(root_dir, do_attr, "final_results.pkl"))
        if loaded is None:
            continue
        scores = evaluate_regression_by_batch(loaded).get(target_attr)
        if scores:
            print(f"do({do_attr})\t{target_attr}: L1-loss = {scores['L1-loss']}")


## FID / minimality artifacts

The `minimality` metric and FID consume the tensors saved under the run's `FID/`
folder. Compute the actual FID with `compute_FID.py`; the cell below just
inspects the saved artifacts.

In [ ]:
import torch
fid_dir = os.path.join(root_dir, "FID")
minimality = torch.load(os.path.join(fid_dir, "minimality.pt"))
counterfactual_images = torch.load(os.path.join(fid_dir, "counterfactual_tensors.pt"))
print("factual feature shape:", np.asarray(minimality['factuals'][1]).shape)
print("num interventions:", len(minimality['interventions']))
print("counterfactual tensor shape:", counterfactual_images.shape)
